# LeetCode 4081 — Prime Subset Selection II (Hard)

Bài toán LeetCode:

```python
class Solution:
    def maxPrimeSubset(self, n: int, s: int) -> List[int]:
        ...
```

Link tra cứu: https://leetcode.com/problemset/?search=4081 (Weekly Contest 523 — câu 4, rating khoảng 2104).

Bài này là một **bài toán cái túi 0/1 (0/1 knapsack)** có ba tầng tiêu chí:

1. tổng **lớn nhất** nhưng không vượt `s`;
2. trong các tổng lớn nhất đó, dùng **ít số nhất**;
3. nếu vẫn hòa, chọn tập (đã sắp tăng) **nhỏ nhất theo thứ tự từ điển**.

Ta sẽ học: **sàng Eratosthenes**, **knapsack 0/1** (và lỗi kinh điển "dùng một món nhiều lần"), cách **xếp chồng nhiều tiêu chí** và cách **truy vết** để ra đúng tập nhỏ nhất theo từ điển.

---

## Sau notebook này, bạn sẽ nắm được

1. Sàng Eratosthenes để liệt kê số nguyên tố `<= n`.
2. Brute Force thử mọi tập con.
3. Knapsack 0/1: tổng nào đạt được, số món ít nhất cho mỗi tổng — và vì sao phải duyệt tổng **từ lớn xuống**.
4. Lưu kèm danh sách để so sánh từ điển (duyệt số nguyên tố giảm dần).
5. DP hậu tố + truy vết tham lam — không cần lưu danh sách.
6. Lỗi hay gặp, mẹo nhận diện, biến thể và bài tập.

# 1. Đề bài yêu cầu gì?

Cho hai số nguyên `n` và `s`. Xét các **số nguyên tố `<= n`**, chọn một tập con của chúng (mỗi số dùng **tối đa một lần**) theo thứ tự ưu tiên:

1. Tổng **lớn nhất có thể** mà **không vượt quá `s`**.
2. Trong các tập có tổng lớn nhất đó, chọn tập có **ít phần tử nhất**.
3. Nếu vẫn hòa, chọn tập mà khi **sắp tăng dần** thì **nhỏ nhất theo thứ tự từ điển**.

Trả về các số đã chọn theo thứ tự tăng dần. Nếu không có tập **khác rỗng** hợp lệ, trả về `[]`.

---

## Ví dụ 1

```python
n = 7, s = 20   ->   [2, 3, 5, 7]
```

Các số nguyên tố `<= 7`: `2, 3, 5, 7`, tổng `17 <= 20` → chọn hết.

## Ví dụ 2

```python
n = 15, s = 18   ->   [5, 13]
```

Số nguyên tố `<= 15`: `2, 3, 5, 7, 11, 13`. Các tập có tổng `18`: `[5, 13]`, `[7, 11]`, `[2, 3, 13]`, `[2, 5, 11]`. Tổng lớn nhất là `18`; ít phần tử nhất là 2 (`[5, 13]` và `[7, 11]`); `[5, 13]` nhỏ hơn theo từ điển vì `5 < 7`.

## Ví dụ 3

```python
n = 5, s = 1   ->   []
```

Số nguyên tố nhỏ nhất là `2 > 1`.

---

## Ràng buộc

```text
1 <= n <= 1000
1 <= s <= 1000
```

Có 168 số nguyên tố `<= 1000`.

# 2. Kiến thức nền tảng

## 2.1. Sàng Eratosthenes

Đánh dấu mọi số từ 2 tới `n` là "có thể nguyên tố". Với mỗi `i` còn được đánh dấu (bắt đầu từ 2), gạch bỏ các bội `i·i, i·i + i, ...` (các bội nhỏ hơn `i·i` đã bị gạch bởi ước nhỏ hơn). Chỉ cần `i <= √n`.

In [1]:
from typing import List


def sieve(n: int) -> List[int]:
    if n < 2:
        return []
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    i = 2
    while i * i <= n:
        if is_prime[i]:
            for j in range(i * i, n + 1, i):
                is_prime[j] = False
        i += 1
    return [x for x in range(n + 1) if is_prime[x]]


print(sieve(15))
print(len(sieve(1000)), "số nguyên tố <= 1000")

[2, 3, 5, 7, 11, 13]
168 số nguyên tố <= 1000


## 2.2. So sánh danh sách theo thứ tự từ điển

Python so sánh `list` (và `tuple`) **theo thứ tự từ điển**: so phần tử đầu, nếu bằng thì so phần tử thứ hai, ... Một `tuple` nhiều tiêu chí như `(-tổng, số_phần_tử, danh_sách)` cho phép so sánh **xếp chồng** các tiêu chí trong một phép `<`.

In [2]:
print([5, 13] < [7, 11])          # True: 5 < 7
print([2, 3, 13] < [5, 13])       # True theo từ điển, nhưng có nhiều phần tử hơn!
key = lambda subset: (-sum(subset), len(subset), subset)
print(min([[5, 13], [7, 11], [2, 3, 13], [2, 5, 11]], key=key))

True
True
[5, 13]


Dòng thứ hai nhắc ta: tiêu chí "ít phần tử" phải được so **trước** tiêu chí từ điển — đúng thứ tự trong `tuple`.

# 3. Cách 1 — Brute Force: thử mọi tập con

In [3]:
from itertools import combinations


def max_prime_subset_bruteforce(n: int, s: int) -> List[int]:
    primes = sieve(n)
    best = None
    for k in range(1, len(primes) + 1):
        for comb in combinations(primes, k):            # combinations giữ thứ tự tăng
            if sum(comb) <= s:
                key = (-sum(comb), k, list(comb))
                if best is None or key < best:
                    best = key
    return [] if best is None else best[2]


print(max_prime_subset_bruteforce(7, 20))    # [2, 3, 5, 7]
print(max_prime_subset_bruteforce(15, 18))   # [5, 13]
print(max_prime_subset_bruteforce(5, 1))     # []

[2, 3, 5, 7]
[5, 13]
[]


`2^168` tập con khi `n = 1000` — không thể. Nhưng `s <= 1000` nhỏ: ta có thể làm DP theo **tổng**.

---

# 4. Knapsack 0/1 từng bước

## 4.1. Tổng nào đạt được?

`can[j] = True` nếu có tập con tổng đúng bằng `j`. Thêm từng số nguyên tố `p`: `can[j] = can[j] or can[j - p]`.

**Quan trọng:** duyệt `j` **từ lớn xuống nhỏ**. Nếu duyệt từ nhỏ lên, `can[j - p]` có thể **đã được cập nhật bằng chính `p`** ở bước này → `p` bị dùng hai lần.

In [4]:
def reachable_sums(primes: List[int], s: int) -> List[bool]:
    can = [False] * (s + 1)
    can[0] = True
    for p in primes:
        for j in range(s, p - 1, -1):          # từ lớn xuống: mỗi p dùng tối đa 1 lần
            if can[j - p]:
                can[j] = True
    return can


can = reachable_sums(sieve(15), 18)
print("tổng đạt được <= 18:", [j for j in range(19) if can[j]])
print("tổng lớn nhất:", max(j for j in range(1, 19) if can[j]))

tổng đạt được <= 18: [0, 2, 3, 5, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18]
tổng lớn nhất: 18


## 4.2. Ít phần tử nhất cho mỗi tổng

`cnt[j]` = số phần tử ít nhất để đạt tổng `j` (∞ nếu không đạt). Chuyển tiếp: `cnt[j] = min(cnt[j], cnt[j - p] + 1)`, vẫn duyệt `j` giảm dần.

In [5]:
INF = float("inf")


def min_count_per_sum(primes: List[int], s: int) -> List[float]:
    cnt = [INF] * (s + 1)
    cnt[0] = 0
    for p in primes:
        for j in range(s, p - 1, -1):
            cnt[j] = min(cnt[j], cnt[j - p] + 1)
    return cnt


cnt = min_count_per_sum(sieve(15), 18)
print({j: cnt[j] for j in range(19) if cnt[j] < INF})

{0: 0, 2: 1, 3: 1, 5: 1, 7: 1, 8: 2, 9: 2, 10: 2, 11: 1, 12: 2, 13: 1, 14: 2, 15: 2, 16: 2, 17: 4, 18: 2}


Tổng 18 cần ít nhất 2 số. Nhưng **2 số nào**? `[5, 13]` hay `[7, 11]`? Ta cần thêm tiêu chí từ điển.

# 5. Cách 2 — Knapsack lưu kèm danh sách

Lưu `dp[j] = (số phần tử, danh sách tăng dần)` tốt nhất cho tổng `j`, so sánh bằng `tuple` (số phần tử trước, rồi từ điển).

Mẹo: duyệt số nguyên tố **giảm dần**. Khi thêm `p`, nó **nhỏ hơn mọi số** trong các danh sách đã có, nên danh sách mới là `[p] + dp[j - p]` — vẫn tăng dần, không cần sắp lại.

Vì sao giữ "tốt nhất" cho từng tổng là đủ? Trong các tập tổng `j` có chứa `p` (số nhỏ nhất), tập tốt nhất là `[p]` + (tập tốt nhất cho tổng `j - p` từ các số lớn hơn): số phần tử ít nhất ⇔ phần còn lại ít nhất; khi đã cùng bắt đầu bằng `p`, so sánh từ điển chỉ còn phụ thuộc phần còn lại.

In [6]:
def max_prime_subset_lists(n: int, s: int) -> List[int]:
    dp = [(INF, [])] * (s + 1)
    dp[0] = (0, [])
    for p in reversed(sieve(n)):                     # số nguyên tố giảm dần
        for j in range(s, p - 1, -1):
            c, lst = dp[j - p]
            if c < INF:
                cand = (c + 1, [p] + lst)
                if cand < dp[j]:
                    dp[j] = cand
    for j in range(s, 0, -1):                         # tổng lớn nhất đạt được (khác rỗng)
        if dp[j][0] < INF:
            return dp[j][1]
    return []


print(max_prime_subset_lists(7, 20))
print(max_prime_subset_lists(15, 18))
print(max_prime_subset_lists(5, 1))

[2, 3, 5, 7]
[5, 13]
[]


Cách 2 đúng và đủ nhanh với `s <= 1000`, nhưng phải tạo nhiều danh sách mới (`[p] + lst`) — tốn bộ nhớ và thời gian sao chép. Cách 3 chỉ lưu **số** rồi **truy vết** danh sách ở cuối.

---

# 6. Cách 3 — DP hậu tố + truy vết tham lam

Đánh số các số nguyên tố tăng dần `primes[0] < primes[1] < ...`. Định nghĩa:

```text
dp[i][j] = số phần tử ít nhất, chỉ dùng primes[i:], để có tổng ĐÚNG BẰNG j   (∞ nếu không được)
dp[P][0] = 0
dp[i][j] = min( dp[i+1][j],  dp[i+1][j - primes[i]] + 1 )
```

1. `target` = `j` lớn nhất (`1 <= j <= s`) có `dp[0][j] < ∞` (tổng lớn nhất).
2. **Truy vết:** đi qua các số nguyên tố theo thứ tự **tăng dần**. Tại `primes[i] = p`, nếu có thể **lấy `p`** mà vẫn hoàn thành tối ưu — tức `dp[i+1][target - p] == dp[i][target] - 1` — thì **lấy** (`target -= p`). Ngược lại bỏ qua.

**Vì sao ra tập nhỏ nhất theo từ điển?** Ở mỗi bước, ta chọn số **nhỏ nhất có thể** làm phần tử tiếp theo của một lời giải tối ưu. Hai tập tăng dần, tập nào có phần tử đầu (khác nhau) nhỏ hơn thì nhỏ hơn theo từ điển → tham lam "lấy sớm nhất có thể" cho kết quả nhỏ nhất.

## Truy vết ví dụ 2 (`n = 15`, `s = 18`)

`primes = [2, 3, 5, 7, 11, 13]`, `target = 18`, `dp[0][18] = 2`.

| i | p | target | cần `dp[i+1][target - p] == dp[i][target] - 1` | lấy? |
|---:|---:|---:|---|:---:|
| 0 | 2 | 18 | `dp[1][16]` = 2 (`3 + 13`) ≠ 1 — lấy 2 thì phải dùng tổng cộng 3 số | |
| 1 | 3 | 18 | `dp[2][15]` = ∞ ≠ 1 | |
| 2 | 5 | 18 | `dp[3][13]` = 1 (số 13) == 1 | ✔ → target 13 |
| 3 | 7 | 13 | `dp[4][6]` = ∞ ≠ 0 | |
| 4 | 11 | 13 | `dp[5][2]` = ∞ ≠ 0 | |
| 5 | 13 | 13 | `dp[6][0]` = 0 == 0 | ✔ → target 0 |

Kết quả `[5, 13]` ✔.

In [7]:
def max_prime_subset_table(n: int, s: int, verbose: bool = False) -> List[int]:
    primes = sieve(n)
    P = len(primes)
    dp = [[INF] * (s + 1) for _ in range(P + 1)]
    dp[P][0] = 0
    for i in range(P - 1, -1, -1):
        p = primes[i]
        for j in range(s + 1):
            dp[i][j] = dp[i + 1][j]
            if j >= p and dp[i + 1][j - p] + 1 < dp[i][j]:
                dp[i][j] = dp[i + 1][j - p] + 1
    target = next((j for j in range(s, 0, -1) if dp[0][j] < INF), 0)
    if verbose:
        print(f"primes = {primes}, tổng lớn nhất = {target}, số phần tử = {dp[0][target] if target else '-'}")
    result = []
    for i in range(P):
        if target == 0:
            break
        p = primes[i]
        take = target >= p and dp[i + 1][target - p] == dp[i][target] - 1
        if verbose:
            need = dp[i + 1][target - p] if target >= p else "—"
            print(f"  p = {p:>2}, target = {target:>2}: dp[i+1][target-p] = {need}, cần {dp[i][target] - 1} -> "
                  f"{'LẤY' if take else 'bỏ qua'}")
        if take:
            result.append(p)
            target -= p
    return result


print(max_prime_subset_table(15, 18, verbose=True))

primes = [2, 3, 5, 7, 11, 13], tổng lớn nhất = 18, số phần tử = 2
  p =  2, target = 18: dp[i+1][target-p] = 2, cần 1 -> bỏ qua
  p =  3, target = 18: dp[i+1][target-p] = inf, cần 1 -> bỏ qua
  p =  5, target = 18: dp[i+1][target-p] = 1, cần 1 -> LẤY
  p =  7, target = 13: dp[i+1][target-p] = inf, cần 0 -> bỏ qua
  p = 11, target = 13: dp[i+1][target-p] = inf, cần 0 -> bỏ qua
  p = 13, target = 13: dp[i+1][target-p] = 0, cần 0 -> LẤY
[5, 13]


# 7. Phiên bản đúng format LeetCode

In [8]:
from typing import List


class Solution:
    def maxPrimeSubset(self, n: int, s: int) -> List[int]:
        if n < 2:
            return []
        is_prime = [True] * (n + 1)
        is_prime[0] = is_prime[1] = False
        i = 2
        while i * i <= n:
            if is_prime[i]:
                for j in range(i * i, n + 1, i):
                    is_prime[j] = False
            i += 1
        primes = [x for x in range(n + 1) if is_prime[x]]

        P = len(primes)
        INF = float("inf")
        dp = [[INF] * (s + 1) for _ in range(P + 1)]
        dp[P][0] = 0
        for i in range(P - 1, -1, -1):
            p = primes[i]
            row, nxt = dp[i], dp[i + 1]
            for j in range(s + 1):
                row[j] = nxt[j]
                if j >= p and nxt[j - p] + 1 < row[j]:
                    row[j] = nxt[j - p] + 1

        target = 0
        for j in range(s, 0, -1):
            if dp[0][j] < INF:
                target = j
                break
        result = []
        for i in range(P):
            if target == 0:
                break
            p = primes[i]
            if target >= p and dp[i + 1][target - p] == dp[i][target] - 1:
                result.append(p)
                target -= p
        return result

In [9]:
sol = Solution()
tests = [
    (7, 20, [2, 3, 5, 7], "ví dụ 1"),
    (15, 18, [5, 13], "ví dụ 2"),
    (5, 1, [], "ví dụ 3"),
    (1, 100, [], "n = 1: không có số nguyên tố"),
    (2, 2, [2], "đúng một số"),
    (3, 4, [3], "không được dùng 2 hai lần"),
    (10, 12, [5, 7], "bài tập 1"),
]
for n, s, expected, name in tests:
    result = sol.maxPrimeSubset(n, s)
    assert result == expected, (name, result, expected)
    print(f"PASS | {name:30} -> {result}")
print("n = 1000, s = 1000 ->", sol.maxPrimeSubset(1000, 1000))

PASS | ví dụ 1                        -> [2, 3, 5, 7]
PASS | ví dụ 2                        -> [5, 13]
PASS | ví dụ 3                        -> []
PASS | n = 1: không có số nguyên tố   -> []
PASS | đúng một số                    -> [2]
PASS | không được dùng 2 hai lần      -> [3]
PASS | bài tập 1                      -> [5, 7]
n = 1000, s = 1000 -> [3, 997]


# 8. Vì sao đúng?

1. **DP hậu tố:** `dp[i][j]` xét hai khả năng cho `primes[i]`: không dùng (`dp[i+1][j]`) hoặc dùng một lần (`dp[i+1][j-p] + 1`). Mỗi số chỉ được xét ở đúng một tầng `i` → dùng tối đa một lần.
2. **Tiêu chí 1 & 2:** `target` là tổng lớn nhất đạt được, `dp[0][target]` là số phần tử ít nhất.
3. **Tiêu chí 3 (truy vết):** khẳng định "tại bước `i`, nếu lấy được `p` mà vẫn hoàn thành tối ưu thì có một lời giải tối ưu nhỏ nhất theo từ điển bắt đầu (phần còn lại) bằng `p`". Đúng vì mọi lời giải tối ưu còn lại chỉ dùng số `>= p`; nếu có lời giải dùng `p` thì nó bắt đầu bằng số nhỏ hơn mọi lời giải không dùng `p`.
4. Điều kiện `dp[i+1][target-p] == dp[i][target] - 1` chính là "lấy `p` rồi phần còn lại vẫn có thể hoàn thành với số phần tử tối thiểu".

# 9. Độ phức tạp

Gọi `P` = số số nguyên tố `<= n` (`P = 168` khi `n = 1000`).

| Cách | Ý tưởng | Thời gian | Bộ nhớ |
|---|---|---:|---:|
| 1. Brute Force | mọi tập con | `O(2^P · P)` | `O(P)` |
| 2. Knapsack + danh sách | `dp[j] = (count, list)` | `O(P · s · L)` (L = độ dài danh sách) | `O(s · L)` |
| 3. DP hậu tố + truy vết | bảng số + tham lam | `O(n log log n + P · s)` | `O(P · s)` |

`P · s ≈ 1.7·10^5` — rất nhỏ.

# 10. Đối chiếu ba cách

In [10]:
for n in range(1, 32):
    for s in range(1, 70):
        expected = max_prime_subset_bruteforce(n, s)
        assert max_prime_subset_lists(n, s) == expected, (n, s)
        assert max_prime_subset_table(n, s) == expected, (n, s)
        assert sol.maxPrimeSubset(n, s) == expected, (n, s)
print("Mọi n < 32, s < 70: 4 hàm cho cùng kết quả!")

Mọi n < 32, s < 70: 4 hàm cho cùng kết quả!


# 11. Những lỗi dễ mắc

## Lỗi 1 — Duyệt tổng từ nhỏ lên (dùng một số nhiều lần)

In [11]:
def wrong_unbounded(n, s):
    dp = [(INF, [])] * (s + 1)
    dp[0] = (0, [])
    for p in reversed(sieve(n)):
        for j in range(p, s + 1):                      # SAI: từ nhỏ lên
            c, lst = dp[j - p]
            if c < INF and (c + 1, [p] + lst) < dp[j]:
                dp[j] = (c + 1, [p] + lst)
    return next((dp[j][1] for j in range(s, 0, -1) if dp[j][0] < INF), [])


print("Sai :", wrong_unbounded(3, 4))
print("Đúng:", sol.maxPrimeSubset(3, 4))

Sai : [2, 2]
Đúng: [3]


Code sai cho `[2, 2]` — dùng số 2 hai lần.

## Lỗi 2 — Tham lam "lấy số lớn nhất còn vừa"

Nghe hợp lý (muốn tổng lớn, ít số) nhưng sai. Tìm phản ví dụ tự động:

In [12]:
def wrong_greedy_largest(n, s):
    res, rem = [], s
    for p in reversed(sieve(n)):
        if p <= rem:
            res.append(p)
            rem -= p
    return sorted(res)


found = None
for n in range(2, 40):
    for s in range(1, 60):
        if wrong_greedy_largest(n, s) != sol.maxPrimeSubset(n, s):
            found = (n, s)
            break
    if found:
        break
print("Phản ví dụ:", found, "| tham lam:", wrong_greedy_largest(*found), "| đúng:", sol.maxPrimeSubset(*found))

Phản ví dụ: (7, 8) | tham lam: [7] | đúng: [3, 5]


## Lỗi 3 — So sánh từ điển trước khi so số phần tử

`[2, 3, 13] < [5, 13]` theo từ điển, nhưng `[2, 3, 13]` có 3 phần tử. Thứ tự tiêu chí phải là: tổng (lớn) → số phần tử (ít) → từ điển (nhỏ).

## Lỗi 4 — Coi tập rỗng là đáp án hợp lệ với tổng 0

Đề yêu cầu tập **khác rỗng**; khi không có, trả `[]`. Khi tìm `target`, chỉ xét `j >= 1`.

## Lỗi 5 — Sàng sai biên

Viết `range(2, n)` sẽ bỏ sót chính `n` khi `n` là số nguyên tố (ví dụ `n = 7`).

In [13]:
def wrong_sieve(n):
    return [x for x in range(2, n) if all(x % d for d in range(2, int(x ** 0.5) + 1))]   # SAI: thiếu n


print("Sai :", wrong_sieve(7), "| Đúng:", sieve(7))

Sai : [2, 3, 5] | Đúng: [2, 3, 5, 7]


# 12. Mẹo nhận diện

1. Chọn tập con, mỗi món **tối đa một lần**, ràng buộc **tổng <= s** với `s` nhỏ → **knapsack 0/1**, duyệt tổng **giảm dần** (hoặc dùng bảng 2 chiều theo món).
2. Nhiều tiêu chí xếp chồng → DP cho tiêu chí chính, **truy vết** cho tiêu chí phụ; hoặc so sánh bằng `tuple`.
3. "Nhỏ nhất theo từ điển" → truy vết theo thứ tự tăng dần, **lấy phần tử sớm nhất có thể** (cần DP **hậu tố** để biết phần còn lại có hoàn thành được không).

Các bài liên quan: *Partition Equal Subset Sum* (LeetCode 416), *Coin Change* (322 — knapsack không giới hạn), *Last Stone Weight II* (1049), *Prime Subset Selection I* (phiên bản dễ hơn trong cùng contest).

# 13. Biến thể

## Biến thể 1 — Đếm số tập con có tổng đúng bằng `s`

In [14]:
def count_subsets_sum(n: int, s: int) -> int:
    ways = [0] * (s + 1)
    ways[0] = 1
    for p in sieve(n):
        for j in range(s, p - 1, -1):
            ways[j] += ways[j - p]
    return ways[s]


print(count_subsets_sum(15, 18))    # [5,13], [7,11], [2,3,13], [2,5,11] -> 4

4


## Biến thể 2 — Nhỏ nhất theo từ điển **mà không cần ít phần tử nhất**

Bỏ tiêu chí 2: chỉ cần tổng lớn nhất rồi từ điển nhỏ nhất. DP chỉ cần lưu "đạt được hay không" cho hậu tố; truy vết: lấy `p` nếu `can[i+1][target - p]`.

In [15]:
def lexmin_max_sum(n: int, s: int) -> List[int]:
    primes = sieve(n)
    P = len(primes)
    can = [[False] * (s + 1) for _ in range(P + 1)]
    can[P][0] = True
    for i in range(P - 1, -1, -1):
        for j in range(s + 1):
            can[i][j] = can[i + 1][j] or (j >= primes[i] and can[i + 1][j - primes[i]])
    target = next((j for j in range(s, 0, -1) if can[0][j]), 0)
    res = []
    for i in range(P):
        if target and target >= primes[i] and can[i + 1][target - primes[i]]:
            res.append(primes[i])
            target -= primes[i]
    return res


print(lexmin_max_sum(15, 18))     # [2, 3, 13]: nhỏ nhất theo từ điển, dù có 3 phần tử

[2, 3, 13]


# 14. Template tư duy

```python
# DP hậu tố cho knapsack 0/1 + truy vết nhỏ nhất theo từ điển
dp[P][0] = 0
for i in reversed(range(P)):
    for j in range(s + 1):
        dp[i][j] = min(dp[i+1][j], dp[i+1][j - w[i]] + 1 if j >= w[i] else INF)
target = tổng tốt nhất
for i in range(P):                       # thứ tự tăng dần
    if target >= w[i] and dp[i+1][target - w[i]] == dp[i][target] - 1:
        lấy w[i]; target -= w[i]
```

# 15. Bài tập vận dụng

## Bài 1 — Tính bằng tay

`n = 10`, `s = 12`. Liệt kê số nguyên tố, tìm tổng lớn nhất, số phần tử ít nhất, và đáp án. Kiểm tra bằng cell dưới.

In [16]:
# Kiểm tra Bài 1
print(max_prime_subset_table(10, 12, verbose=True))

primes = [2, 3, 5, 7], tổng lớn nhất = 12, số phần tử = 2
  p =  2, target = 12: dp[i+1][target-p] = 2, cần 1 -> bỏ qua
  p =  3, target = 12: dp[i+1][target-p] = inf, cần 1 -> bỏ qua
  p =  5, target = 12: dp[i+1][target-p] = 1, cần 1 -> LẤY
  p =  7, target =  7: dp[i+1][target-p] = 0, cần 0 -> LẤY
[5, 7]


## Bài 2 — Dự đoán output

```python
print(sieve(30))
print([1, 9] < [2], (2, [5]) < (2, [3, 4]), (1, [9]) < (2, [1]))
```

In [17]:
print(sieve(30))
print([1, 9] < [2], (2, [5]) < (2, [3, 4]), (1, [9]) < (2, [1]))

[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
True False True


## Bài 3 — Tự viết sàng Eratosthenes

In [18]:
def exercise_3_sieve(n: int) -> List[int]:
    # TODO
    raise NotImplementedError("Hãy tự hoàn thiện trước khi xem đáp án.")


# assert exercise_3_sieve(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
# assert exercise_3_sieve(1) == []

## Bài 4 — Partition Equal Subset Sum (LeetCode 416)

Có chia được mảng thành hai phần có tổng bằng nhau không?

```python
can_partition([1, 5, 11, 5])   -> True
can_partition([1, 2, 3, 5])    -> False
```

In [19]:
def exercise_4_can_partition(nums: List[int]) -> bool:
    # TODO
    raise NotImplementedError


# assert exercise_4_can_partition([1, 5, 11, 5]) == True
# assert exercise_4_can_partition([1, 2, 3, 5]) == False

## Bài 5 — Số phần tử ít nhất để có tổng đúng bằng `s`

Trả về số số nguyên tố ít nhất (mỗi số dùng một lần) có tổng **đúng bằng** `s`, hoặc `-1`.

```python
min_primes_exact(15, 18)   -> 2
min_primes_exact(5, 1)     -> -1
min_primes_exact(10, 17)   -> 4     # 2 + 3 + 5 + 7 (không có cách nào ít hơn)
```

In [20]:
def exercise_5_min_primes_exact(n: int, s: int) -> int:
    # TODO
    raise NotImplementedError


# assert exercise_5_min_primes_exact(15, 18) == 2
# assert exercise_5_min_primes_exact(5, 1) == -1

## Bài 6 — Mini challenge: lớn nhất theo từ điển

Giữ tiêu chí 1 và 2, nhưng đổi tiêu chí 3 thành: tập (sắp tăng) **lớn nhất** theo từ điển. Gợi ý: khi truy vết tăng dần, **ưu tiên bỏ qua** số nhỏ nếu vẫn hoàn thành tối ưu được.

```python
max_prime_subset_lexmax(15, 18)   -> [7, 11]
```

In [21]:
def exercise_6_lexmax(n: int, s: int) -> List[int]:
    # TODO
    raise NotImplementedError


# assert exercise_6_lexmax(15, 18) == [7, 11]

## Gợi ý (mở từng mức)

**Bài 3**

- Mức 1: mảng `is_prime` dài `n + 1`, đặt `0`, `1` là `False`.
- Mức 2: với `i` từ 2 khi `i * i <= n`: nếu `is_prime[i]` thì gạch `range(i*i, n+1, i)`.
- Mức 3: trả về các `x` còn `True`; xử lý `n < 2`.

**Bài 4**

- Mức 1: tổng lẻ → `False`.
- Mức 2: knapsack bool với mục tiêu `tổng / 2`.
- Mức 3: duyệt tổng giảm dần.

**Bài 5**

- Mức 1: `cnt[j]` = số phần tử ít nhất cho tổng đúng `j`.
- Mức 2: `cnt[j] = min(cnt[j], cnt[j - p] + 1)`, `j` giảm dần.
- Mức 3: trả `cnt[s]` hoặc `-1`.

**Bài 6**

- Mức 1: vẫn dùng bảng `dp` hậu tố.
- Mức 2: tại `p`, nếu **bỏ qua** vẫn hoàn thành tối ưu (`dp[i+1][target] == dp[i][target]`) thì bỏ qua.
- Mức 3: ngược lại bắt buộc lấy `p`.

# 16. Đáp án tham khảo

## Đáp án Bài 1

Số nguyên tố `<= 10`: `2, 3, 5, 7`. Tổng `12` đạt được bằng `5 + 7` (2 số) hoặc `2 + 3 + 7` (3 số). Đáp án `[5, 7]`.

## Đáp án Bài 2

```text
[2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
True False True
```

In [22]:
# Đáp án Bài 3
def answer_3_sieve(n: int) -> List[int]:
    if n < 2:
        return []
    is_prime = [True] * (n + 1)
    is_prime[0] = is_prime[1] = False
    i = 2
    while i * i <= n:
        if is_prime[i]:
            for j in range(i * i, n + 1, i):
                is_prime[j] = False
        i += 1
    return [x for x in range(n + 1) if is_prime[x]]


assert answer_3_sieve(30) == [2, 3, 5, 7, 11, 13, 17, 19, 23, 29]
assert answer_3_sieve(1) == []
print("Bài 3: PASS")

Bài 3: PASS


In [23]:
# Đáp án Bài 4
def answer_4_can_partition(nums: List[int]) -> bool:
    total = sum(nums)
    if total % 2:
        return False
    half = total // 2
    can = [False] * (half + 1)
    can[0] = True
    for x in nums:
        for j in range(half, x - 1, -1):
            can[j] = can[j] or can[j - x]
    return can[half]


assert answer_4_can_partition([1, 5, 11, 5]) == True
assert answer_4_can_partition([1, 2, 3, 5]) == False
print("Bài 4: PASS")

Bài 4: PASS


In [24]:
# Đáp án Bài 5
def answer_5_min_primes_exact(n: int, s: int) -> int:
    cnt = min_count_per_sum(sieve(n), s)
    return cnt[s] if cnt[s] < INF else -1


assert answer_5_min_primes_exact(15, 18) == 2
assert answer_5_min_primes_exact(5, 1) == -1
assert answer_5_min_primes_exact(10, 17) == 4
print("Bài 5: PASS")

Bài 5: PASS


In [25]:
# Đáp án Bài 6
def answer_6_lexmax(n: int, s: int) -> List[int]:
    primes = sieve(n)
    P = len(primes)
    dp = [[INF] * (s + 1) for _ in range(P + 1)]
    dp[P][0] = 0
    for i in range(P - 1, -1, -1):
        for j in range(s + 1):
            dp[i][j] = dp[i + 1][j]
            if j >= primes[i]:
                dp[i][j] = min(dp[i][j], dp[i + 1][j - primes[i]] + 1)
    target = next((j for j in range(s, 0, -1) if dp[0][j] < INF), 0)
    res = []
    for i in range(P):
        if target == 0:
            break
        if dp[i + 1][target] == dp[i][target]:
            continue                                      # bỏ qua được -> bỏ qua
        res.append(primes[i])
        target -= primes[i]
    return res


def brute_lexmax(n, s):
    best = None
    ps = sieve(n)
    for k in range(1, len(ps) + 1):
        for c in combinations(ps, k):
            if sum(c) <= s:
                key = (-sum(c), k)
                if best is None or key < best[0] or (key == best[0] and list(c) > best[1]):
                    best = (key, list(c))
    return [] if best is None else best[1]


assert answer_6_lexmax(15, 18) == [7, 11]
for n in range(1, 25):
    for s in range(1, 50):
        assert answer_6_lexmax(n, s) == brute_lexmax(n, s), (n, s)
print("Bài 6: PASS (khớp brute force với mọi n < 25, s < 50)")

Bài 6: PASS (khớp brute force với mọi n < 25, s < 50)


# 17. Tổng kết

**Prime Subset Selection II**:

1. Sàng Eratosthenes → danh sách số nguyên tố.
2. Knapsack 0/1 theo tổng (`s <= 1000`), mỗi số dùng một lần (duyệt tổng giảm dần, hoặc bảng theo số).
3. Ba tiêu chí: tổng lớn nhất → ít số nhất (giá trị DP) → nhỏ nhất theo từ điển (truy vết tăng dần, lấy sớm nhất có thể).

## Độ phức tạp

```text
Time:  O(n log log n + P · s)
Space: O(P · s)
```

# 18. Checklist trước khi submit

- [ ] Sàng bao gồm cả `n` (`range(..., n + 1)`)?
- [ ] Mỗi số nguyên tố dùng **tối đa một lần** (tổng giảm dần / bảng theo số)?
- [ ] Thứ tự tiêu chí: tổng → số phần tử → từ điển?
- [ ] Chỉ xét tổng `>= 1` (tập khác rỗng)?
- [ ] Truy vết theo thứ tự **tăng dần** với DP **hậu tố**?
- [ ] Đã test: `n = 1`, `s` nhỏ hơn 2, chọn hết, có nhiều tập hòa?